In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [9]:
years=["2017","2018","2019","2020","2021","2022","2023","2024","2025"]
objective=["summary","player_stats"]

In [18]:
# Cargamos los CSV

# Esta es la ruta de los archivos CSV
ruta_archivos_csv = r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode"
# Listamos los archivos CSV

archivos_csv_summary = [os.path.join(ruta_archivos_csv, archivo) for archivo in os.listdir(ruta_archivos_csv) if archivo.endswith(".csv") and "summary" in archivo and "2022" in archivo]
archivos_csv_players = [os.path.join(ruta_archivos_csv, archivo) for archivo in os.listdir(ruta_archivos_csv) if archivo.endswith(".csv") and "Player_Stats" in archivo and "2022" in archivo]

In [19]:
# Veamos que archivos se tienen

print("Cantidad de archivos sumamry que se cargarán: ", len(archivos_csv_summary))
print("Cantidad de archivos players que se cargarán: ", len(archivos_csv_players))
#for archivo in archivos_csv:
    #print(os.path.basename(archivo).length)

Cantidad de archivos sumamry que se cargarán:  8684
Cantidad de archivos players que se cargarán:  1168


In [5]:
# Reacomodamos los nombres

mapa_columnas = {
"Player":["Unnamed: 0_level_0 Player"],
"Number":["Unnamed: 1_level_0 #"],
"Country":["Unnamed: 2_level_0 Nation"],
"Position":["Unnamed: 3_level_0 Pos"],
"Age":["Unnamed:4_level_0 Age"],
"Min":["Unnamed: 5_level_0 Min"],
"Goals":["Performance Gls"],
"Assists":["Performance Ast"],
"GoalsPen":["Performance PK"],
"AttPen":["Performance PKatt"],
"Shots":["Performance Sh"],
"ShTarget":["Performance SoT"],
"YellowCard":["Performance CrdY"],
"RedCard":["Performance CrdR"],
"Fouls":["Performance Fls"],
"Offside":["Performance Off"],
"Crosses":["Performance Crs"],
"TacklesW":["Performance TklW"],
"Interceptions":["Performance Int"],
"OwnGoals":["Performance OG"],
"PenKickGen":["Performance PKwon"],
"PenKickCon":["Performance PKcon"]
}

In [6]:
print(pd.read_csv(archivos_csv_summary[1]).head())
test=pd.read_csv(archivos_csv_summary[1]).head()

   _table_index               _table_id                  _table_name  \
0             1  stats_06c1606c_summary  Banfield Player Stats Table   
1             1  stats_06c1606c_summary  Banfield Player Stats Table   
2             1  stats_06c1606c_summary  Banfield Player Stats Table   
3             1  stats_06c1606c_summary  Banfield Player Stats Table   
4             1  stats_06c1606c_summary  Banfield Player Stats Table   

                                            match_id        date  \
0  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
1  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
2  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
3  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   
4  Barracas-Central-Banfield-February-1-2025-Liga...  2025-02-01   

                                league      team     kind  \
0  ar Liga Profesional Argentina Table  06c1606c  summary   
1  ar Liga Profesional Argentina Tab

In [ ]:
# Consolidar CSVs en archivos limpios (summary y player_stats) por ano
import re
import hashlib
from pathlib import Path

# Configura si deseas sobreescribir salidas existentes
OVERWRITE_OUTPUTS = True

def detect_file_type(filename: str, year: str) -> str:
    name = filename.lower()
    if year not in name:
        return "unknown"
    if "player" in name and "stats" in name:
        return "player_stats"
    if "summary" in name:
        return "summary"
    return "unknown"

def parse_match_meta(filename: str) -> dict:
    base = Path(filename).name
    name = base.rsplit(".", 1)[0]
    left = name.split("__")[0]
    m = re.search(r"(\d{2}-\d{2}-\d{4})", left)
    date = m.group(1) if m else ""
    competition = ""
    home = ""
    away = ""
    if m:
        date_start = left.find(date)
        if date_start > 0:
            competition = left[: date_start - 1]
        after = left[date_start + len(date) + 1 :] if date_start >= 0 else ""
        if "-vs_" in after:
            home, away = after.split("-vs_", 1)
    match_key = f"{competition}|{date}|{home}|{away}"
    match_id = hashlib.md5(match_key.encode("utf-8")).hexdigest() if match_key != "|||" else ""
    return {
        "match_id": match_id,
        "match_date": date,
        "home_team": home,
        "away_team": away,
        "competition": competition,
    }

def read_csv_safe(path: str, nrows=None):
    try:
        return pd.read_csv(path, nrows=nrows)
    except UnicodeDecodeError:
        return pd.read_csv(path, nrows=nrows, encoding="latin-1")

def collect_columns(file_list):
    cols = set()
    errors = []
    for i, f in enumerate(file_list, 1):
        try:
            df = read_csv_safe(f, nrows=0)
            cols.update(df.columns.tolist())
        except Exception as exc:
            errors.append((f, str(exc)))
        if i % 1000 == 0:
            print(f"Columnas escaneadas en {i} archivos")
    return cols, errors

def extract_team_from_table_name(series: pd.Series) -> pd.Series:
    return series.astype(str).str.replace(r"\s*Player Stats Table$", "", regex=True).str.strip()

def write_consolidated(file_list, out_path, cols, file_type):
    meta_cols = ["match_id", "match_date", "home_team", "away_team", "competition", "source_file", "file_type", "team"]
    final_cols = meta_cols + sorted(cols)
    if out_path.exists() and OVERWRITE_OUTPUTS:
        out_path.unlink()
    wrote_header = out_path.exists()
    manifest_rows = []
    for i, f in enumerate(file_list, 1):
        info = parse_match_meta(f)
        row = {"source_file": f, "file_type": file_type, **info}
        try:
            df = read_csv_safe(f)
            for k, v in info.items():
                df[k] = v
            if "_table_name" in df.columns:
                df["team"] = extract_team_from_table_name(df["_table_name"])
            else:
                df["team"] = ""
            df["source_file"] = f
            df["file_type"] = file_type
            df = df.reindex(columns=final_cols)
            df.to_csv(out_path, mode="a", index=False, header=not wrote_header)
            wrote_header = True
            row["rows"] = len(df)
            row["columns"] = len(df.columns)
            row["error"] = ""
        except Exception as exc:
            row["rows"] = 0
            row["columns"] = 0
            row["error"] = str(exc)
        manifest_rows.append(row)
        if i % 500 == 0:
            print(f"{file_type}: procesados {i} archivos")
    return manifest_rows

all_csv = [str(p) for p in Path(ruta_archivos_csv).glob("*.csv")]
print("Total CSV:", len(all_csv))

for year in years:
    out_summary = Path(ruta_archivos_csv) / f"summary_consolidado_{year}.csv"
    out_players = Path(ruta_archivos_csv) / f"players_consolidado_{year}.csv"
    out_manifest = Path(ruta_archivos_csv) / f"manifest_consolidacion_{year}.csv"

    summary_files = [f for f in all_csv if detect_file_type(f, year) == "summary"]
    player_files = [f for f in all_csv if detect_file_type(f, year) == "player_stats"]
    print("\nAno:", year)
    print("Summary:", len(summary_files))
    print("Player stats:", len(player_files))

    summary_cols, summary_col_errors = collect_columns(summary_files)
    player_cols, player_col_errors = collect_columns(player_files)
    print("Columnas summary:", sorted(summary_cols))
    print("Columnas player_stats:", sorted(player_cols))

    manifest = []
    manifest += write_consolidated(summary_files, out_summary, summary_cols, "summary")
    manifest += write_consolidated(player_files, out_players, player_cols, "player_stats")

    manifest_df = pd.DataFrame(manifest)
    if out_manifest.exists() and OVERWRITE_OUTPUTS:
        out_manifest.unlink()
    manifest_df.to_csv(out_manifest, index=False)
    print("Listo. Salidas:", out_summary, out_players, out_manifest)

Total CSV: 224034

Ano: 2017
Summary: 4177
Player stats: 0
Columnas escaneadas en 1000 archivos
Columnas escaneadas en 2000 archivos
Columnas escaneadas en 3000 archivos
Columnas escaneadas en 4000 archivos
Columnas summary: ['Carries Carries', 'Carries PrgC', 'Expected npxG', 'Expected xAG', 'Expected xG', 'Passes Att', 'Passes Cmp', 'Passes Cmp%', 'Passes PrgP', 'Performance Ast', 'Performance Blocks', 'Performance CrdR', 'Performance CrdY', 'Performance Crs', 'Performance Fld', 'Performance Fls', 'Performance Gls', 'Performance Int', 'Performance OG', 'Performance Off', 'Performance PK', 'Performance PKatt', 'Performance PKcon', 'Performance PKwon', 'Performance Sh', 'Performance SoT', 'Performance Tkl', 'Performance TklW', 'Performance Touches', 'SCA GCA', 'SCA SCA', 'Take-Ons Att', 'Take-Ons Succ', 'Unnamed: 0_level_0 Player', 'Unnamed: 1_level_0 #', 'Unnamed: 2_level_0 Nation', 'Unnamed: 3_level_0 Pos', 'Unnamed: 4_level_0 Age', 'Unnamed: 5_level_0 Min', '_table_id', '_table_inde

C:\Users\ruben\AppData\Local\Temp\ipykernel_9396\152106459.py:47: DtypeWarning: Columns (43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, nrows=nrows)


player_stats: procesados 500 archivos
Listo. Salidas: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\summary_consolidado_2021.csv C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\players_consolidado_2021.csv C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\manifest_consolidacion_2021.csv

Ano: 2022
Summary: 8685
Player stats: 1168
Columnas escaneadas en 1000 archivos


In [ ]:
# Concat vertical de consolidados por ano (summary, players, manifest)
from pathlib import Path

base = Path(ruta_archivos_csv)
summary_out = base / "summary_consolidado.csv"
players_out = base / "players_consolidado.csv"
manifest_out = base / "manifest_consolidacion.csv"

def concat_year_files(prefix: str, out_path: Path):
    frames = []
    for year in years:
        src = base / f"{prefix}_{year}.csv"
        if src.exists():
            frames.append(pd.read_csv(src))
    if frames:
        df = pd.concat(frames, axis=0, ignore_index=True, sort=False)
        df.to_csv(out_path, index=False)
        print("Listo:", out_path, "filas:", len(df))
    else:
        print("No se encontraron archivos para", prefix)

concat_year_files("summary_consolidado", summary_out)
concat_year_files("players_consolidado", players_out)
concat_year_files("manifest_consolidacion", manifest_out)

In [ ]:
# Revision rapida de consolidados (head, columnas, nulos, duplicados por match/equipo, errores)
from pathlib import Path

base = Path(ruta_archivos_csv)
manifest_path = base / "manifest_consolidacion.csv"
players_path = base / "players_consolidado.csv"
summary_path = base / "summary_consolidado.csv"

def detect_team_column(columns):
    candidates = ["Squad", "Team", "Club", "Equipo", "team", "squad"]
    for c in candidates:
        if c in columns:
            return c
    return None

def review_manifest(path: Path):
    print("\n=== MANIFEST ===")
    if not path.exists():
        print("No existe:", path)
        return
    df = pd.read_csv(path)
    print("Filas:", len(df), "Columnas:", len(df.columns))
    if "error" in df.columns:
        err = df[df["error"].astype(str).str.strip() != ""]
        print("Archivos con error:", len(err))
        if len(err) > 0:
            print(err.head(20))
    else:
        print("No hay columna 'error' en el manifest")

def review_csv(path: Path):
    print("\n===", path.name, "===")
    if not path.exists():
        print("No existe:", path)
        return
    # Head y columnas
    head_df = pd.read_csv(path, nrows=3)
    print(head_df)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    print("Columnas:", cols)
    # Estadisticas por chunks
    row_count = 0
    null_counts = pd.Series(0, index=cols, dtype="int64")
    team_col = detect_team_column(cols)
    key_cols = ["match_id", team_col] if ("match_id" in cols and team_col) else None
    dup_key_count = 0
    seen_keys = set()
    for chunk in pd.read_csv(path, chunksize=200000):
        row_count += len(chunk)
        null_counts = null_counts.add(chunk.isna().sum(), fill_value=0)
        if key_cols:
            keys = chunk[key_cols].astype(str).agg("||".join, axis=1)
            dup_key_count += keys.isin(seen_keys).sum()
            seen_keys.update(keys.unique().tolist())
    print("Filas:", row_count, "Columnas:", len(cols))
    print("Top nulos:")
    print(null_counts.sort_values(ascending=False).head(20))
    if key_cols:
        print("Duplicados por", key_cols, ":", int(dup_key_count))
    else:
        print("No se detecto match_id + equipo para duplicados")

review_manifest(manifest_path)
review_csv(players_path)
review_csv(summary_path)


=== MANIFEST ===
Filas: 82089 Columnas: 10
Archivos con error: 82089
                                          source_file file_type  \
0   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
1   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
2   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
3   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
4   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
5   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
6   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
7   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
8   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
9   C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
10  C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
11  C:\Users\ruben\Documents\Universidad\TEG\Codig...   summary   
12  C:\Users\ruben\Documents\Universidad\TEG\Codig...   sum

In [25]:
# Aplanar a 1 fila por equipo/partido (desde player_stats consolidado)
from pathlib import Path

players_path = Path(ruta_archivos_csv) / "players_consolidado.csv"
out_team_match = Path(ruta_archivos_csv) / "players_team_match_agg.csv"

if not players_path.exists():
    raise FileNotFoundError(f"No existe {players_path}. Ejecuta la celda de consolidacion primero.")

# Columnas de jugador que se excluyen del agregado
player_only_cols = {
    "Player", "Number", "Country", "Position", "Age", "Min"
}

def detect_team_column(columns):
    candidates = ["Squad", "Team", "Club", "Equipo", "team", "squad"]
    for c in candidates:
        if c in columns:
            return c
    return None

chunk_iter = pd.read_csv(players_path, chunksize=200000)
agg_frames = []
team_col = None
for i, chunk in enumerate(chunk_iter, 1):
    if team_col is None:
        team_col = detect_team_column(chunk.columns)
        if team_col is None:
            raise ValueError("No se encontro columna de equipo (Squad/Team/Club).")
    # Filtra columnas no numericas y excluye campos de jugador
    keep_cols = [c for c in chunk.columns if c not in player_only_cols]
    chunk = chunk[keep_cols]
    # Coacciona a numerico lo que se pueda
    numeric_cols = chunk.select_dtypes(include=["number"]).columns.tolist()
    for c in chunk.columns:
        if c not in numeric_cols and c not in {"match_id", "match_date", "home_team", "away_team", "competition", "source_file", "file_type", team_col}:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
    numeric_cols = chunk.select_dtypes(include=["number"]).columns.tolist()
    group_cols = ["match_id", team_col]
    # Suma metricas numericas por equipo/partido
    agg = chunk.groupby(group_cols, as_index=False)[numeric_cols].sum()
    # Agrega metadatos del partido (primer valor no nulo)
    meta_cols = [c for c in ["match_date", "home_team", "away_team", "competition"] if c in chunk.columns]
    if meta_cols:
        meta = chunk[group_cols + meta_cols].drop_duplicates(subset=group_cols).groupby(group_cols, as_index=False).first()
        agg = agg.merge(meta, on=group_cols, how="left")
    agg_frames.append(agg)
    if i % 10 == 0:
        print(f"Chunks procesados: {i}")

# Re-agrupa al final para eliminar duplicados entre chunks
agg_all = pd.concat(agg_frames, ignore_index=True)
numeric_cols_all = agg_all.select_dtypes(include=["number"]).columns.tolist()
group_cols_all = ["match_id", team_col]
final_agg = agg_all.groupby(group_cols_all, as_index=False)[numeric_cols_all].sum()

meta_cols_all = [c for c in ["match_date", "home_team", "away_team", "competition"] if c in agg_all.columns]
if meta_cols_all:
    meta_all = agg_all[group_cols_all + meta_cols_all].drop_duplicates(subset=group_cols_all).groupby(group_cols_all, as_index=False).first()
    final_agg = final_agg.merge(meta_all, on=group_cols_all, how="left")

final_agg.to_csv(out_team_match, index=False)
print("Listo. Salida:", out_team_match)

ValueError: No se encontro columna de equipo (Squad/Team/Club).

In [ ]:
# Aplanar a 1 fila por equipo/partido (desde summary consolidado)
from pathlib import Path

summary_path = Path(ruta_archivos_csv) / "summary_consolidado.csv"
out_summary_team_match = Path(ruta_archivos_csv) / "summary_team_match_agg.csv"

if not summary_path.exists():
    raise FileNotFoundError(f"No existe {summary_path}. Ejecuta la celda de consolidacion primero.")

def detect_team_column(columns):
    candidates = ["Squad", "Team", "Club", "Equipo", "team", "squad"]
    for c in candidates:
        if c in columns:
            return c
    return None

chunk_iter = pd.read_csv(summary_path, chunksize=200000)
agg_frames = []
team_col = None
for i, chunk in enumerate(chunk_iter, 1):
    if team_col is None:
        team_col = detect_team_column(chunk.columns)
        if team_col is None:
            raise ValueError("No se encontro columna de equipo (Squad/Team/Club).")
    # Coacciona a numerico lo que se pueda
    numeric_cols = chunk.select_dtypes(include=["number"]).columns.tolist()
    for c in chunk.columns:
        if c not in numeric_cols and c not in {"match_id", "match_date", "home_team", "away_team", "competition", "source_file", "file_type", team_col}:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
    numeric_cols = chunk.select_dtypes(include=["number"]).columns.tolist()
    group_cols = ["match_id", team_col]
    # Suma metricas numericas por equipo/partido
    agg = chunk.groupby(group_cols, as_index=False)[numeric_cols].sum()
    # Agrega metadatos del partido (primer valor no nulo)
    meta_cols = [c for c in ["match_date", "home_team", "away_team", "competition"] if c in chunk.columns]
    if meta_cols:
        meta = chunk[group_cols + meta_cols].drop_duplicates(subset=group_cols).groupby(group_cols, as_index=False).first()
        agg = agg.merge(meta, on=group_cols, how="left")
    agg_frames.append(agg)
    if i % 10 == 0:
        print(f"Chunks procesados: {i}")

# Re-agrupa al final para eliminar duplicados entre chunks
agg_all = pd.concat(agg_frames, ignore_index=True)
numeric_cols_all = agg_all.select_dtypes(include=["number"]).columns.tolist()
group_cols_all = ["match_id", team_col]
final_agg = agg_all.groupby(group_cols_all, as_index=False)[numeric_cols_all].sum()

meta_cols_all = [c for c in ["match_date", "home_team", "away_team", "competition"] if c in agg_all.columns]
if meta_cols_all:
    meta_all = agg_all[group_cols_all + meta_cols_all].drop_duplicates(subset=group_cols_all).groupby(group_cols_all, as_index=False).first()
    final_agg = final_agg.merge(meta_all, on=group_cols_all, how="left")

final_agg.to_csv(out_summary_team_match, index=False)
print("Listo. Salida:", out_summary_team_match)

In [ ]:
# Concat de summary y players a nivel equipo/partido
from pathlib import Path

players_team_path = Path(ruta_archivos_csv) / "players_team_match_agg.csv"
summary_team_path = Path(ruta_archivos_csv) / "summary_team_match_agg.csv"
out_concat = Path(ruta_archivos_csv) / "team_match_concat.csv"

if not players_team_path.exists():
    raise FileNotFoundError(f"No existe {players_team_path}. Ejecuta la celda 8 primero.")
if not summary_team_path.exists():
    raise FileNotFoundError(f"No existe {summary_team_path}. Ejecuta la celda 9 primero.")

df_players = pd.read_csv(players_team_path)
df_summary = pd.read_csv(summary_team_path)

df_concat = pd.concat([df_players, df_summary], axis=0, ignore_index=True, sort=False)
df_concat.to_csv(out_concat, index=False)
print("Listo. Salida:", out_concat)

In [17]:
# Buscar CSV cuyo nombre de columna contenga "AEK Athens (4-1-3-2)"
from pathlib import Path

target_col_substring = "AEK Athens (4-1-3-2)"
YearZ = ["2017","2018","2019","2020","2021","2022","2023","2024","2025"]  # Ajusta el ano aqui
base = Path(ruta_archivos_csv)
csv_files = [p for p in base.glob("*.csv") if any(y in p.name for y in YearZ)]

matches = []
for f in csv_files:
    try:
        cols = pd.read_csv(f, nrows=0).columns.tolist()
    except UnicodeDecodeError:
        cols = pd.read_csv(f, nrows=0, encoding="latin-1").columns.tolist()
    if any(target_col_substring in c for c in cols):
        matches.append(str(f))

print("Archivos con columna que contiene el texto:")
for m in matches:
    print(m)
print("Total:", len(matches))

KeyboardInterrupt: 

In [ ]:
# Normalizar columnas por tipo y por 
# ano (players, summary, manifest)
from pathlib import Path

base = Path(ruta_archivos_csv)
types = {
    "players": "players_consolidado",
    "summary": "summary_consolidado",
    "manifest": "manifest_consolidacion",
}

columnas_unicas_players = set()
columnas_unicas_summary = set()
columnas_unicas_manifest = set()
cols_map = {
    "players": columnas_unicas_players,
    "summary": columnas_unicas_summary,
    "manifest": columnas_unicas_manifest,
}

def read_cols(path: Path):
    try:
        return pd.read_csv(path, nrows=0).columns.tolist()
    except UnicodeDecodeError:
        return pd.read_csv(path, nrows=0, encoding="latin-1").columns.tolist()

# 1) Recolectar columnas unicas por tipo
for year in years:
    for key, prefix in types.items():
        src = base / f"{prefix}_{year}.csv"
        if src.exists():
            cols_map[key].update(read_cols(src))

print("Columnas unicas players:", sorted(columnas_unicas_players))
print("Columnas unicas summary:", sorted(columnas_unicas_summary))
print("Columnas unicas manifest:", sorted(columnas_unicas_manifest))

# 2) Normalizar cada CSV anual y guardar con sufijo _sort
def normalize_and_save(prefix: str, cols_set: set):
    final_cols = sorted(cols_set)
    for year in years:
        src = base / f"{prefix}_{year}.csv"
        if not src.exists():
            continue
        try:
            df = pd.read_csv(src)
        except UnicodeDecodeError:
            df = pd.read_csv(src, encoding="latin-1")
        # Agregar columnas faltantes
        for c in final_cols:
            if c not in df.columns:
                df[c] = pd.NA
        # Ordenar columnas y guardar
        df = df[final_cols]
        out_path = base / f"{prefix}_{year}_sort.csv"
        df.to_csv(out_path, index=False)
        print("Listo:", out_path)

normalize_and_save("players_consolidado", columnas_unicas_players)
normalize_and_save("summary_consolidado", columnas_unicas_summary)
normalize_and_save("manifest_consolidacion", columnas_unicas_manifest)

In [ ]:
# Mantener solo columnas comunes entre players_consolidado y summary_consolidado
from pathlib import Path

base = Path(ruta_archivos_csv)
players_path = base / "players_consolidado.csv"
summary_path = base / "summary_consolidado.csv"

if not players_path.exists() or not summary_path.exists():
    raise FileNotFoundError("Faltan players_consolidado.csv o summary_consolidado.csv")

# Usa las columnas unicas si existen, si no, lee headers
if "columnas_unicas_players" in globals() and "columnas_unicas_summary" in globals():
    common_cols = sorted(set(columnas_unicas_players) & set(columnas_unicas_summary))
else:
    players_cols = pd.read_csv(players_path, nrows=0).columns.tolist()
    summary_cols = pd.read_csv(summary_path, nrows=0).columns.tolist()
    common_cols = sorted(set(players_cols) & set(summary_cols))

print("Columnas comunes:", common_cols)

df_players = pd.read_csv(players_path)
df_summary = pd.read_csv(summary_path)

df_players = df_players[common_cols]
df_summary = df_summary[common_cols]

df_players.to_csv(players_path, index=False)
df_summary.to_csv(summary_path, index=False)
print("Listo. Sobrescritos:", players_path, summary_path)